In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# os.chdir removed — run from workflows/01_tobermorite_optimization/notebooks/


In [ ]:
# === 1. Load Data ===
df_a = pd.read_csv('../../../data/training/training_data_123.csv')
df = df_a.dropna()

X = df.values[:, :4]
y = df.values[:, 4:]

# === 2. Split into train and test ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === 3. Scale ===
X_scaler = StandardScaler().fit(X_train)
y_scaler = StandardScaler().fit(y_train)

X_train_scaled = X_scaler.transform(X_train)
y_train_scaled = y_scaler.transform(y_train)

X_test_scaled = X_scaler.transform(X_test)
y_test_scaled = y_scaler.transform(y_test)


In [ ]:
# === 1. Define Flexible Neural Network ===
class FlexibleNN(nn.Module):
    def __init__(self, input_dim, hidden_layers, output_dim, dropout_p=0.5):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_layers:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(p=dropout_p))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# === 2. Train & Select Best Model ===
def train_and_select_best_model(X_train_scaled, y_train_scaled, X_test_scaled, y_test_scaled,
                                 hidden_layers, dropout_p=0.5, lr=1e-3, weight_decay=1e-5,
                                 n_epochs=1000, n_trials=10):
    X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
        X_train_scaled, y_train_scaled, test_size=0.2, random_state=42)

    best_model = None
    best_val_mse = float('inf')
    best_state_dict = None

    for trial in range(n_trials):
        model = FlexibleNN(input_dim=4, hidden_layers=hidden_layers, output_dim=9, dropout_p=dropout_p)
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
        criterion = nn.MSELoss()

        X_tensor = torch.tensor(X_train_split, dtype=torch.float32)
        y_tensor = torch.tensor(y_train_split, dtype=torch.float32)

        for epoch in range(n_epochs):
            model.train()
            optimizer.zero_grad()
            output = model(X_tensor)
            loss = criterion(output, y_tensor)
            loss.backward()
            optimizer.step()

        # Validation performance
        model.eval()
        with torch.no_grad():
            X_val_tensor = torch.tensor(X_val_split, dtype=torch.float32)
            y_val_pred_scaled = model(X_val_tensor).numpy()
            y_val_pred = y_scaler.inverse_transform(y_val_pred_scaled)
            y_val_real = y_scaler.inverse_transform(y_val_split)
            val_mse = mean_squared_error(y_val_real, y_val_pred)

        print(f"[Trial {trial+1}] Val MSE: {val_mse:.4f}")

        if val_mse < best_val_mse:
            best_val_mse = val_mse
            best_state_dict = model.state_dict()

    best_model = FlexibleNN(4, hidden_layers, 9, dropout_p)
    best_model.load_state_dict(best_state_dict)
    print(f"\n✅ Best model selected with Val MSE: {best_val_mse:.4f}")
    return best_model


def evaluate_model(model, X_scaled, y_scaled, y_true=None, dataset_name="Set"):
    model.eval()
    with torch.no_grad():
        X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
        y_pred_scaled = model(X_tensor).numpy()
        y_pred = y_scaler.inverse_transform(y_pred_scaled)
        y_true = y_scaler.inverse_transform(y_scaled) if y_true is None else y_true

        mse = mean_squared_error(y_true, y_pred)
        mae = mean_absolute_error(y_true, y_pred)
        r2 = r2_score(y_true, y_pred)

        print(f"\n--- {dataset_name} Performance ---")
        print(f"MSE: {mse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
        
        return y_true, y_pred


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 64, 32],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 64, 32],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-3,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 64, 32],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-5,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 64, 32],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[16],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[256],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[28],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[64],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[64, 64, 64, 64, 64],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 128, 128, 128, 128],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=1e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,           # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 128, 128],  # customize this
    dropout_p=0.2,                # try 0.2, 0.3, 0.5
    lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=0,               # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 128, 128],  # customize this
    dropout_p=0.5,                # try 0.2, 0.3, 0.5
    lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,               # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[128, 128, 128],  # customize this
    dropout_p=0.6,                # try 0.2, 0.3, 0.5
    lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-5,               # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[64, 32],  # customize this
    dropout_p=0.5,                # try 0.2, 0.3, 0.5
    lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-5,               # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
model = train_and_select_best_model(
    X_train_scaled, y_train_scaled,
    X_test_scaled, y_test_scaled,
    hidden_layers=[64, 32],  # customize this
    dropout_p=0.3,                # try 0.2, 0.3, 0.5
    lr=5e-4,                      # try 1e-3, 5e-4, 1e-4
    weight_decay=1e-4,               # L2 regularization
    n_epochs=1000,
    n_trials=10
)

# === 5. Final Evaluation ===
y_train_real, y_train_pred = evaluate_model(model, X_train_scaled, y_train_scaled, y_train, "Train")
y_test_real, y_test_pred = evaluate_model(model, X_test_scaled, y_test_scaled, y_test, "Test")


In [ ]:
import matplotlib.pyplot as plt
prop_dict = {}
    
for i, prop in enumerate(['D_11', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14']):
    prop_dict[f'Property {i+1}'] = prop

# === 6. Visualize ===
fig, axes = plt.subplots(3, 3, figsize=(16, 14))
axes = axes.flatten()
for i in range(9):
    r2_train = r2_score(y_train_real[:, i], y_train_pred[:, i])
    r2_test = r2_score(y_test_real[:, i], y_test_pred[:, i])
    axes[i].scatter(y_train_real[:, i], y_train_pred[:, i], alpha=0.6, color='blue', label=f'Train (R²={r2_train:.2f})')
    axes[i].scatter(y_test_real[:, i], y_test_pred[:, i], alpha=0.6, color='red', label=f'Test (R²={r2_test:.2f})')
    axes[i].plot([min(y_train_real[:, i].min(), y_test_real[:, i].min()), 
                  max(y_train_real[:, i].max(), y_test_real[:, i].max())],
                 [min(y_train_real[:, i].min(), y_test_real[:, i].min()), 
                  max(y_train_real[:, i].max(), y_test_real[:, i].max())], 'k--')
    target_prop = [2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47]
    axes[i].axvline(target_prop[i], color='green', linestyle='--', marker='None', linewidth=2, label='Target Property')
    axes[i].set_title(prop_dict[f"Property {i+1}"])

    axes[i].set_xlabel("True")
    axes[i].set_ylabel("Predicted")
    axes[i].legend()

plt.tight_layout()
plt.show()



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os

# === 0. Load Data ===
# os.chdir removed — run from workflows/01_tobermorite_optimization/notebooks/
df = pd.read_csv('../../../data/training/training_data_123.csv').dropna()
X = df.values[:, :4]
y = df.values[:, 4:]

# === 1. Hold out 10% for final test ===
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === 2. Define Model ===
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 9)
        )

    def forward(self, x):
        return self.net(x)

# === 3. Train Function ===
def train_model(X_train, y_train, X_val, y_val, l2_lambda=1e-4, n_epochs=1000):
    model = SimpleNN()
    optimizer = optim.Adam(model.parameters(), lr=5e-4, weight_decay=l2_lambda)
    criterion = nn.MSELoss()

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype=torch.float32)

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        output = model(X_train_tensor)
        loss = criterion(output, y_train_tensor)
        loss.backward()
        optimizer.step()

    # Validation loss
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_tensor).numpy()
        val_true = y_val_tensor.numpy()
        val_loss = mean_squared_error(val_true, val_pred)

    return model, val_loss

# === 4. 5-Fold Cross Validation on Train+Val ===
kf = KFold(n_splits=5, shuffle=True, random_state=42)
val_mse_scores, val_mae_scores, val_r2_scores = [], [], []

best_model = None
best_val_loss = float('inf')

fold = 1
for train_idx, val_idx in kf.split(X_trainval):
    X_train_fold, X_val_fold = X_trainval[train_idx], X_trainval[val_idx]
    y_train_fold, y_val_fold = y_trainval[train_idx], y_trainval[val_idx]

    # Scale within fold
    X_scaler = StandardScaler().fit(X_train_fold)
    y_scaler = StandardScaler().fit(y_train_fold)

    X_train_scaled = X_scaler.transform(X_train_fold)
    y_train_scaled = y_scaler.transform(y_train_fold)
    X_val_scaled = X_scaler.transform(X_val_fold)
    y_val_scaled = y_scaler.transform(y_val_fold)

    model, val_loss = train_model(X_train_scaled, y_train_scaled, X_val_scaled, y_val_scaled)

    # Evaluate
    model.eval()
    with torch.no_grad():
        y_val_pred_scaled = model(torch.tensor(X_val_scaled, dtype=torch.float32)).numpy()
        y_val_pred = y_scaler.inverse_transform(y_val_pred_scaled)
        y_val_true = y_val_fold

        mse = mean_squared_error(y_val_true, y_val_pred)
        mae = mean_absolute_error(y_val_true, y_val_pred)
        r2 = r2_score(y_val_true, y_val_pred)

        val_mse_scores.append(mse)
        val_mae_scores.append(mae)
        val_r2_scores.append(r2)

        print(f"[Fold {fold}] Val MSE: {mse:.2f}, MAE: {mae:.2f}, R²: {r2:.4f}")
        fold += 1

        if mse < best_val_loss:
            best_val_loss = mse
            best_model = model
            best_scaler_X = X_scaler
            best_scaler_y = y_scaler

# === 5. Report Validation Results ===
print("\n=== 5-Fold Cross-Validation Summary ===")
print(f"Avg Val MSE: {np.mean(val_mse_scores):.2f} ± {np.std(val_mse_scores):.2f}")
print(f"Avg Val MAE: {np.mean(val_mae_scores):.2f} ± {np.std(val_mae_scores):.2f}")
print(f"Avg Val R²:  {np.mean(val_r2_scores):.4f} ± {np.std(val_r2_scores):.4f}")

# === 6. Final Evaluation on Held-out Test Set ===
X_test_scaled = best_scaler_X.transform(X_test)
y_test_scaled = best_scaler_y.transform(y_test)

with torch.no_grad():
    y_test_pred_scaled = best_model(torch.tensor(X_test_scaled, dtype=torch.float32)).numpy()
    y_test_pred = best_scaler_y.inverse_transform(y_test_pred_scaled)

test_mse = mean_squared_error(y_test, y_test_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("\n=== Held-out Test Set Performance ===")
print(f"Test MSE: {test_mse:.2f}")
print(f"Test MAE: {test_mae:.2f}")
print(f"Test R²:  {test_r2:.4f}")

# === 7. Save the Best Model ===
torch.save(best_model.state_dict(), "best_model.pt")


In [ ]:
# === 4. Optimization Function ===
def inverse_design(model, target_props):
    model.eval()

    def objective(x_flat):
        x_tensor = torch.tensor(x_flat.reshape(1, -1), dtype=torch.float32)
        with torch.no_grad():
            pred_scaled = model(x_tensor).numpy()
        target_scaled = y_scaler.transform([target_props])
        return np.mean((pred_scaled - target_scaled) ** 2)

    x0 = np.mean(X_train_scaled, axis=0)
    bounds = [(None, None)] * X.shape[1]  # You can customize bounds here if needed
    result = minimize(objective, x0, method='L-BFGS-B', bounds=bounds)

    x_opt_scaled = result.x
    x_opt_real = X_scaler.inverse_transform([x_opt_scaled])[0]

    print("\n✅ Inverse design completed.")
    print("Optimized input parameters (real scale):", x_opt_real)
    return x_opt_real


target_y = [2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47]
x_opt = inverse_design(model, target_y)
x_opt

In [ ]:
import torch
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from botorch.acquisition import LogExpectedImprovement
from botorch.optim import optimize_acqf
from gpytorch.mlls import ExactMarginalLogLikelihood

def inverse_design_bo(model, target_props, X_train, X_scaler, y_scaler, n_init=10, n_iter=25):
    model.eval()
    input_dim = X_train.shape[1]

    # === Target property scaled ===
    target_scaled = torch.tensor(y_scaler.transform([target_props]), dtype=torch.float64)

    # === Create MinMaxScaler on standard-scaled input ===
    X_train_std = X_scaler.transform(X_train)
    minmax_scaler = MinMaxScaler()
    X_train_minmax = minmax_scaler.fit_transform(X_train_std)

    # === Initial samples in [0, 1] ===
    X_init_minmax = torch.tensor(
        np.random.uniform(0, 1, size=(n_init, input_dim)),
        dtype=torch.float64
    )

    def evaluate_model(x_minmax_tensor):
        x_std_np = minmax_scaler.inverse_transform(x_minmax_tensor.cpu().numpy())
        x_tensor_std = torch.tensor(x_std_np, dtype=torch.float32)  # model expects float32
        with torch.no_grad():
            y_pred = model(x_tensor_std).detach()
        return y_pred.to(dtype=torch.float64)

    # === Evaluate initial samples ===
    Y_init_model = evaluate_model(X_init_minmax)
    Y_obj = torch.mean((Y_init_model - target_scaled) ** 2, dim=1, keepdim=True)

    X_train_bo = X_init_minmax.clone()
    Y_train_bo = Y_obj.clone()

    bounds_bo = torch.tensor([[0.0] * input_dim, [1.0] * input_dim], dtype=torch.float64)

    for i in range(n_iter):
        # 1. Fit GP model
        gp = SingleTaskGP(X_train_bo, Y_train_bo)
        mll = ExactMarginalLogLikelihood(gp.likelihood, gp)
        fit_gpytorch_mll(mll)

        # 2. Acquisition function
        acq_func = LogExpectedImprovement(gp, best_f=Y_train_bo.min().item())

        # 3. Optimize acquisition function
        candidate, _ = optimize_acqf(
            acq_function=acq_func,
            bounds=bounds_bo,
            q=1,
            num_restarts=10,
            raw_samples=100,
        )

        # 4. Evaluate new point
        y_new_model = evaluate_model(candidate)
        y_obj = torch.mean((y_new_model - target_scaled) ** 2, dim=1, keepdim=True)

        # 5. Update dataset
        X_train_bo = torch.cat([X_train_bo, candidate], dim=0)
        Y_train_bo = torch.cat([Y_train_bo, y_obj], dim=0)

        print(f"Iter {i+1:02d}: MSE = {y_obj.item():.4f}")

    # === Final result ===
    best_idx = torch.argmin(Y_train_bo)
    best_input_minmax = X_train_bo[best_idx].numpy()
    best_input_std = minmax_scaler.inverse_transform([best_input_minmax])
    best_input_real = X_scaler.inverse_transform(best_input_std)[0]

    print("\n✅ Bayesian optimization complete.")
    print("Optimized input (real scale):", best_input_real)
    return best_input_real


target_y = [2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47]
x_opt_bo = inverse_design_bo(
    model=model,
    target_props=target_y,
    X_train=X_train,
    X_scaler=X_scaler,
    y_scaler=y_scaler,
    n_init=10,
    n_iter=25
)

# ✅ Bayesian optimization complete.
# Optimized input (real scale): [4.58264616 0.27870944 3.24561794 0.11637735]

# ✅ Bayesian optimization complete.
# Optimized input (real scale): [4.49524453 0.40799847 3.52211589 0.16455023]

# ✅ Bayesian optimization complete.
# Optimized input (real scale): [4.2076174  0.295171   3.57162071 0.15522409]

In [ ]:
import os
from mdsetup import MDSetup
import pandas as pd
import numpy as np

os.chdir('')
lammps_setup = MDSetup(
        system_setup="../config/setup_mechanical_pcff.yaml",
        simulation_default="../config/defaults.yaml",
        simulation_ensemble="../config/ensemble.yaml",
        simulation_sampling="../config/sampling_mechanical.yaml",
        submission_command="qsub",
    )

LJ_SETS = [f"LJ_set_124"]

def MD_sim(suggested_params):
    TOB_STRUCTURES = ["Tob11", "Tob11H", "Tob14"]
    # 1. Density Analysis
    den_results = {tob: [] for tob in TOB_STRUCTURES}

    for lj_set in LJ_SETS:
        for tob_structure in TOB_STRUCTURES:
            analysis_folder = f"{tob_structure}/{lj_set}/equilibration"    
            extracted_values = lammps_setup.analysis_extract_properties(
                analysis_folder=analysis_folder,
                ensemble='01_npt',
                extracted_properties=['density'],
                output_suffix='density',
                time_fraction=0.2,
            )
            average_values = extracted_values.get('01_npt', {}).get("data", {}).get("average", {})
            mean_value = average_values.get('density', {}).get("mean", None)
            den_results[tob_structure].append(mean_value)

    # 2. Surface Energy Analysis
    NA = 6.022e23
    CONVERSION = 4184  # kcal/mol to J/mol

    # Box dimensions (structure-dependent)
    box_coords = {
        "Tob11":    [0.527972175, 23.057572175, -0.431033519, 21.723966481],
        "Tob11H":   [0.065633394, 22.383633394, -0.317852898, 29.242147102],
        "Tob14":    [-0.658063909, 21.871536091, -0.289451837, 21.985548163],
    }

    se_results = {struct: [] for struct in TOB_STRUCTURES}
    for struct in TOB_STRUCTURES:
        for lj in LJ_SETS:
            try:
                base = f"{struct}/{lj}/SE"
                
                state = 'bulk'
                folder = f"{base}/{state}"
                extracted_values = lammps_setup.analysis_extract_properties(
                    analysis_folder=folder,
                    ensemble="00_nvt",
                    extracted_properties=["potential energy"],
                    output_suffix="energy",
                    time_fraction=0.4,
                )
                bulk_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})

                state = 'vacuum'
                folder = f"{base}/{state}"
                extracted_values = lammps_setup.analysis_extract_properties(
                    analysis_folder=folder,
                    ensemble="00_nvt",
                    extracted_properties=["potential energy"],
                    output_suffix="energy",
                    time_fraction=0.4,
                )
                vac_energy = extracted_values.get("00_nvt", {}).get("data", {}).get("average", {}).get("potential energy", {})
                
                E_bulk, SD_bulk = bulk_energy["mean"], bulk_energy["std"]
                E_vac, SD_vac = vac_energy["mean"], vac_energy["std"]

                # Compute surface area
                xlo, xhi, ylo, yhi = box_coords[struct]
                A = abs(xhi - xlo) * abs(yhi - ylo) * 1e-20  # m²

                # Compute SE and std dev
                deltaE = 1000 * (E_vac - E_bulk) * CONVERSION / (2 * A * NA)  # mJ/m²
                s_dev = 1000 * (SD_bulk + SD_vac) * CONVERSION / (2 * A * NA)

                se_results[struct].append((deltaE, s_dev))
                print(f"✅ Analyzed {struct} with {lj}: SE = {deltaE:.2f} ± {s_dev:.2f} mJ/m²")
            
            except Exception as e:
                print(f"❌ Failed to analyze {struct} with {lj}: {e}")
                se_results[struct].append((None, None))
                continue
    
    # 3. Mechanical Properties Analysis            
    bm_results = {tob: [] for tob in TOB_STRUCTURES}
    for lj_set in LJ_SETS:
        for tob_structure in TOB_STRUCTURES:
            try:
                # Define the deformation analysis folder
                analysis_folder = f"{tob_structure}/{lj_set}/deformation"
                # Run analysis
                BM, Cij = lammps_setup.analysis_mechanical_proerties(
                    analysis_folder=analysis_folder,
                    ensemble="00_nvt",
                    deformation_rates=[-0.02, -0.01, 0.00, 0.01, 0.02],
                    method="VRH",
                    time_fraction=0.4,
                    visualize_stress_strain=False,
                )

                print(f"✅ Analyzed {tob_structure} with {lj_set}: BM = {BM:.2f} GPa")
                bm_results[tob_structure].append(BM)

            except Exception as e:
                print(f"❌ Failed to analyze {tob_structure} with {lj_set}: {e}")
                bm_results[tob_structure].append(None)
                continue

    sc_r, sc_eps, oc_r, oc_eps =  suggested_params[:, 0], suggested_params[:, 1], suggested_params[:, 2], suggested_params[:, 3]
    
    
    df = pd.DataFrame({
        'sc_r': sc_r,
        'sc_eps': sc_eps,
        'oc_r': oc_r,
        'oc_eps': oc_eps,
        
        'D_11': np.array(den_results['Tob11']),
        'D_11H': np.array(den_results['Tob11H']),
        'D_14': np.array(den_results['Tob14']),
        
        'SE_T11': np.array(se_results['Tob11'])[:,0],
        'SE_T11H': np.array(se_results['Tob11H'])[:,0],
        'SE_T14': np.array(se_results['Tob14'])[:,0],
        
        'BM_T11': np.array(bm_results['Tob11']),
        'BM_T11H': np.array(bm_results['Tob11H']),
        'BM_T14': np.array(bm_results['Tob14'])
        
    })

    print("✅ All jobs completed and data saved.")
    return df


In [ ]:
x_opt_bo = np.array([[4.74310034, 0.24331045, 3.14598884, 0.10200781]])

In [ ]:
# from run_md_sim import MD_sim
df_new = MD_sim(np.atleast_2d(x_opt_bo))

In [ ]:
target_y = np.array([2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47])
df_new.values[:, 4:] - target_y

In [ ]:
md_prop = df_new.values[:,4:]

traget_prop = np.array(target_y)

diff = 100*(md_prop - traget_prop)/traget_prop

dddd1 = df_new[['sc_r', 'sc_eps', 'oc_r', 'oc_eps']]
dddd2 = pd.DataFrame(diff.round(2), columns=['D_11(error %)', 'D_11H', 'D_14', 'SE_T11', 'SE_T11H', 'SE_T14', 'BM_T11', 'BM_T11H', 'BM_T14'])

dd_final = pd.concat([dddd1, dddd2], axis=1)
# save dd_final into csv file
# dd_final.to_csv('ml_results5.csv', index=False)

In [ ]:
dd_final

In [ ]:
dd_final = pd.read_csv('ml_results5.csv')

In [ ]:
df_new = df[-5:].copy()

In [ ]:
target_y = np.array([2.46, 2.39, 2.23, 680, 325, 635, 71, 55.35, 47])

md_prop_pred = df_new.values[:,4:]


y_error = np.abs(100*(md_prop_pred - np.array(target_y))/np.array(target_y))

for i in range(len(y_error)):
    targeted_error = np.array([1, 1, 1, 5, 5, 5, 10, 10, 10])
    if (y_error[i] <= targeted_error).sum() == 9:
        print("All properties are within error limits")

    else:
        print("❌ Failure! The predicted properties are outside the acceptable error range.")
        total_false = (~(y_error[i] <= targeted_error)).sum()
        print(f"Failed for {total_false} properties")
        print("Error percentage:", y_error[i].round(2))
    